# Campagne FPGA uniquement

Notebook autonome (independant) pour executer uniquement la methode `fpga`.

In [5]:
# ==========================================================
# CELLULE 1 - CONFIGURATION (AUTONOME)
# ==========================================================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False

try:
    from ipywidgets import Dropdown, IntText, SelectMultiple, VBox, Label, Output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

try:
    from pynq import Overlay, allocate
    HAS_PYNQ = True
except Exception:
    HAS_PYNQ = False
    Overlay = None
    allocate = None

FS_DEFAULT = 11_999_000.0
IF_DEFAULT = 3_563_000.0
N_DEFAULT = 11999
NB_PHASES_DEFAULT = 1023
EPS = 1e-12

DDS_LUT_SIZE = 1024
DDS_PHASE_BITS = 32
DDS_LUT_BITS = 10

HEADER_RE = re.compile(
    r"PRN=(?P<prn>-?\d+)"
    r".*SNR=(?P<snr>-?\d+(?:\.\d+)?)dB"
    r".*Doppler=(?P<doppler>-?\d+(?:\.\d+)?)Hz"
    r".*CA_SHIFT=(?P<ca>-?\d+)chips",
    re.IGNORECASE,
)

METHOD_LABELS = {
    "cpu_lut_angles": "CPU LUT angles",
    "fpga": "FPGA",
}

EXPORT_DIR_DEFAULT = "Resultats validation SA"

def _methods_export_tag(cfg):
    methods = [str(m) for m in cfg.get("run_methods", []) if str(m)]
    if not methods:
        return "no_method"
    return "_vs_".join(methods)


def resolve_export_dir(cfg):
    base = Path(str(cfg.get("export_dir", EXPORT_DIR_DEFAULT)))
    if bool(cfg.get("export_methods_subdir", True)):
        return base / _methods_export_tag(cfg)
    return base


DEFAULT_CONFIG = {
    "signals_dir": "signals",
    "prn_dir": "PRN",
    "signal_glob": "*.csv",
    "limit_signals": None,
    "selected_signal_files": None,
    "run_methods": ["fpga"],
    "prn_list": None,
    "fs_hz": FS_DEFAULT,
    "if_hz": IF_DEFAULT,
    "n_samples": N_DEFAULT,
    "nb_phases": NB_PHASES_DEFAULT,
    "fd_start": -10000,
    "fd_end": 10000,
    "fd_step": 500,
    "detection_ratio_min": 2.5,
    "winner_margin_db_min": 3.0,
    "doppler_tol_hz": 500,
    "phase_tol_chip": 3,
    "bit_path": "sadfsnr.bit",
    "prn_mode": "all_available",
    "prn_search_mode": "injected_plus_decoys",
    "pfa_decoy_count": 8,
    "pfa_decoy_seed": 42,
    "resume_from_partial": False,
    "export_prefix": "validation_fpga",
    "export_dir": "Resultats validation SA/fpga_only",
    "export_methods_subdir": False,
    "autosave_every_acq": 10,
}


def collect_signal_files_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    files = sorted(base_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            with file_path.open("r", encoding="utf-8", errors="ignore") as handle:
                first = handle.readline().strip()
            if HEADER_RE.search(first):
                valid_files.append(file_path.name)
        except Exception:
            continue
    return valid_files


def get_available_prns_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in base_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


WIDGET_STATE = {
    "enabled": False,
    "signal_mode": None,
    "limit_value": None,
    "signal_selector": None,
    "fd_mode": None,
    "fd_start": None,
    "fd_end": None,
    "fd_step": None,
    "prn_mode": None,
    "prn_selector": None,
    "prn_search_mode": None,
}


def get_current_config():
    cfg = dict(DEFAULT_CONFIG)
    if not WIDGET_STATE["enabled"]:
        return cfg

    signal_mode = int(WIDGET_STATE["signal_mode"].value)
    if signal_mode == 0:
        cfg["limit_signals"] = None
        cfg["selected_signal_files"] = None
    elif signal_mode == 1:
        cfg["limit_signals"] = int(WIDGET_STATE["limit_value"].value)
        cfg["selected_signal_files"] = None
    else:
        cfg["limit_signals"] = None
        cfg["selected_signal_files"] = [str(WIDGET_STATE["signal_selector"].value)] if WIDGET_STATE["signal_selector"].value else None

    if int(WIDGET_STATE["fd_mode"].value) == 0:
        cfg["fd_start"] = -10000
        cfg["fd_end"] = 10000
        cfg["fd_step"] = 500
    else:
        cfg["fd_start"] = int(WIDGET_STATE["fd_start"].value)
        cfg["fd_end"] = int(WIDGET_STATE["fd_end"].value)
        cfg["fd_step"] = int(WIDGET_STATE["fd_step"].value)

    if int(WIDGET_STATE["prn_mode"].value) == 0:
        cfg["prn_list"] = None
    else:
        cfg["prn_list"] = list(WIDGET_STATE["prn_selector"].value)

    psm = int(WIDGET_STATE["prn_search_mode"].value)
    cfg["prn_search_mode"] = "injected_only" if psm == 0 else ("injected_plus_decoys" if psm == 1 else "full_grid")
    return cfg


_available_signals = collect_signal_files_for_ui(DEFAULT_CONFIG["signals_dir"])
_available_prns = get_available_prns_for_ui(DEFAULT_CONFIG["prn_dir"])

print("=== CONFIGURATION INTERACTIVE ===")
print(f"Signaux trouves: {len(_available_signals)}")
print(f"PRN disponibles: {_available_prns}")

if HAS_WIDGETS and HAS_DISPLAY and _available_prns:
    signal_mode = Dropdown(
        options=[("Tous", 0), ("Limiter", 1), ("Fichier precis", 2)],
        value=0,
        description="Signaux",
    )
    limit_value = IntText(value=min(10, max(1, len(_available_signals))), description="Nb")
    signal_selector = Dropdown(
        options=[("-- choisir --", "")] + [(name, name) for name in _available_signals],
        value="",
        description="Fichier",
    )
    fd_mode = Dropdown(options=[("Standard", 0), ("Personnalise", 1)], value=0, description="Doppler")
    fd_start = IntText(value=DEFAULT_CONFIG["fd_start"], description="Fd debut")
    fd_end = IntText(value=DEFAULT_CONFIG["fd_end"], description="Fd fin")
    fd_step = IntText(value=DEFAULT_CONFIG["fd_step"], description="Fd step")
    prn_mode = Dropdown(options=[("Tous PRN", 0), ("Selection", 1)], value=0, description="PRN")
    prn_selector = SelectMultiple(options=[(str(p), p) for p in _available_prns], value=tuple(_available_prns[:1]), rows=min(10, len(_available_prns)), description="Liste")
    prn_search_mode = Dropdown(
        options=[("Injecte seulement", 0), ("Injecte + leurres", 1), ("Grille complete", 2)],
        value=1,
        description="Recherche",
    )
    config_output = Output()

    def _refresh(_=None):
        cfg = get_current_config()
        with config_output:
            config_output.clear_output()
            print("=== CONFIG EFFECTIVE (selon menu) ===")
            print(pd.Series(cfg, dtype="object"))

    def _on_limit_value_change(change):
        if int(signal_mode.value) == 0 and change.get("new") is not None:
            signal_mode.value = 1

    def _on_signal_selector_change(change):
        if change.get("new"):
            signal_mode.value = 2

    def _on_prn_selector_change(change):
        if int(prn_mode.value) == 0 and change.get("new"):
            prn_mode.value = 1

    def _on_fd_custom_change(change):
        if int(fd_mode.value) == 0:
            fd_mode.value = 1

    for w in [signal_mode, limit_value, signal_selector, fd_mode, fd_start, fd_end, fd_step, prn_mode, prn_selector, prn_search_mode]:
        w.observe(_refresh, names="value")

    limit_value.observe(_on_limit_value_change, names="value")
    signal_selector.observe(_on_signal_selector_change, names="value")
    prn_selector.observe(_on_prn_selector_change, names="value")
    fd_start.observe(_on_fd_custom_change, names="value")
    fd_end.observe(_on_fd_custom_change, names="value")
    fd_step.observe(_on_fd_custom_change, names="value")

    WIDGET_STATE.update({
        "enabled": True,
        "signal_mode": signal_mode,
        "limit_value": limit_value,
        "signal_selector": signal_selector,
        "fd_mode": fd_mode,
        "fd_start": fd_start,
        "fd_end": fd_end,
        "fd_step": fd_step,
        "prn_mode": prn_mode,
        "prn_selector": prn_selector,
        "prn_search_mode": prn_search_mode,
    })

    display(VBox([
        Label("Choix de campagne FPGA"),
        signal_mode, limit_value, signal_selector,
        fd_mode, fd_start, fd_end, fd_step,
        prn_mode, prn_selector,
        prn_search_mode,
        config_output,
    ]))
    _refresh()
else:
    print("[INFO] Widgets non disponibles ou PRN introuvables: configuration par defaut utilisee.")

CONFIG = get_current_config()
print()
print("[INFO] La configuration effective se met a jour dans le panneau interactif ci-dessus.")

=== CONFIGURATION INTERACTIVE ===
Signaux trouves: 86
PRN disponibles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]



[INFO] La configuration effective se met a jour dans le panneau interactif ci-dessus.


In [6]:
# ==========================================================
# CELLULE 2 - CAMPAGNE DE VALIDATION FPGA
# ==========================================================
FPGA_CONTEXT = None


def to_s32(x):
    x = int(x)
    return x if x < 0x80000000 else x - 0x100000000


def phase_error_mod(measured, expected, modulo=1023):
    d = (int(measured) - int(expected)) % int(modulo)
    return int(min(d, modulo - d))


def parse_meta_from_header(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        first = handle.readline().strip()
    match = HEADER_RE.search(first)
    if not match:
        raise ValueError(f"Metadonnees introuvables dans {csv_file.name}")
    return {
        "prn": int(match.group("prn")),
        "snr": float(match.group("snr")),
        "doppler": int(round(float(match.group("doppler")))),
        "ca_shift": int(match.group("ca")),
    }


def load_signal_pm1(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        line1 = handle.readline().strip()
        line2 = handle.readline().strip()
    if line1 and not line1.startswith("#") and not line2:
        line2 = line1
    if not line2:
        raise ValueError(f"Signal vide: {csv_file}")
    data = np.fromstring(line2, sep=",", dtype=np.float32)
    if data.size == 0:
        raise ValueError(f"Format invalide: {csv_file}")
    data = np.where(data == 255, -1, data).astype(np.float32)
    return data


def load_prn_sequence(prn_dir, prn, n_samples):
    prn_dir = Path(prn_dir)
    candidates = [
        prn_dir / f"PRN-{prn}.csv",
        prn_dir / f"prn-{prn}.csv",
        prn_dir / f"PRN_{prn}.csv",
        prn_dir / f"prn_{prn}.csv",
    ]
    prn_path = next((path for path in candidates if path.exists()), None)
    if prn_path is None:
        raise FileNotFoundError(f"PRN {prn} introuvable dans {prn_dir}")
    seq = np.genfromtxt(prn_path, delimiter=",", dtype=np.float64)
    seq = np.atleast_1d(seq).reshape(-1)
    if seq.size < n_samples:
        raise ValueError(f"PRN trop court ({seq.size}) dans {prn_path}, attendu >= {n_samples}")
    return seq[:n_samples].astype(np.float32)


def get_available_prns(prn_dir):
    prn_dir = Path(prn_dir)
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in prn_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


def collect_signal_files(signals_dir, signal_glob):
    signals_dir = Path(signals_dir)
    files = sorted(signals_dir.glob(signal_glob)) if signals_dir.exists() else []
    if not files and signals_dir.exists():
        files = sorted(signals_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            parse_meta_from_header(file_path)
            valid_files.append(file_path)
        except Exception:
            continue
    return valid_files


def get_fpga_context(cfg):
    global FPGA_CONTEXT
    if FPGA_CONTEXT is not None:
        return FPGA_CONTEXT
    if not HAS_PYNQ:
        raise RuntimeError("pynq indisponible dans ce kernel")
    overlay = Overlay(cfg["bit_path"], download=True)
    FPGA_CONTEXT = {
        "overlay": overlay,
        "ip": getattr(overlay, "acquisition_serial_0"),
        "rx_dma": getattr(overlay, "axi_dma_0"),
        "prn_dma": getattr(overlay, "axi_dma_1"),
        "out_dma": getattr(overlay, "axi_dma_2"),
    }
    return FPGA_CONTEXT


def _dma_channel_is_idle(ch):
    try:
        return bool(getattr(ch, "idle"))
    except Exception:
        return True


def _hard_reset_dma(dma):
    """Reset hardware AXI DMA via MMIO (bit 2 du DMACR)."""
    if dma is None:
        return
    try:
        mmio = getattr(dma, "mmio", None) or getattr(dma, "_mmio", None)
        if mmio is None:
            return
        mmio.write(0x00, 0x04)
        mmio.write(0x30, 0x04)
        for _ in range(100):
            cr_tx = mmio.read(0x00) & 0x04
            cr_rx = mmio.read(0x30) & 0x04
            if cr_tx == 0 and cr_rx == 0:
                break
            time.sleep(0.001)
        mmio.write(0x00, 0x01)
        mmio.write(0x30, 0x01)
    except Exception:
        pass


def _recover_dma_and_ip(ctx, cfg=None, reload_overlay=False):
    global FPGA_CONTEXT
    try:
        ctx["ip"].write(0x00, 0x00)
    except Exception:
        pass

    for key in ("rx_dma", "prn_dma", "out_dma"):
        _hard_reset_dma(ctx.get(key))

    time.sleep(0.05)

    if reload_overlay and cfg is not None:
        try:
            overlay = Overlay(cfg["bit_path"], download=True)
            FPGA_CONTEXT = {
                "overlay": overlay,
                "ip": getattr(overlay, "acquisition_serial_0"),
                "rx_dma": getattr(overlay, "axi_dma_0"),
                "prn_dma": getattr(overlay, "axi_dma_1"),
                "out_dma": getattr(overlay, "axi_dma_2"),
            }
            ctx.update(FPGA_CONTEXT)
            print("[DMA-RECOVERY] overlay recharge")
            time.sleep(0.1)
        except Exception as reload_exc:
            print(f"[DMA-RECOVERY] echec reload overlay: {reload_exc}")


def fpga_acquisition(signal, prn, cfg):
    t_total0 = time.perf_counter()

    ctx = get_fpga_context(cfg)
    n_samples = int(cfg["n_samples"])
    nb_phases = int(cfg["nb_phases"])
    fd_start = int(cfg["fd_start"])
    fd_end = int(cfg["fd_end"])
    fd_step = int(cfg["fd_step"])
    ratio_min = float(cfg["detection_ratio_min"])
    nb_fd = (fd_end - fd_start) // fd_step + 1
    total_outputs = nb_fd * nb_phases

    in_sig = allocate(shape=(n_samples,), dtype=np.int32)
    in_prn = allocate(shape=(n_samples,), dtype=np.int32)
    out_buf = allocate(shape=(total_outputs,), dtype=np.int32)

    in_sig[:] = np.rint(np.real(np.asarray(signal[:n_samples]))).astype(np.int32)
    in_prn[:] = np.rint(np.real(np.asarray(prn[:n_samples]))).astype(np.int32)
    out_buf[:] = 0
    prep_done = time.perf_counter()

    try:
        try:
            ctx["ip"].register_map.fd_step = int(fd_step)
        except Exception:
            ctx["ip"].write(0x40, int(fd_step))

        last_exc = None
        max_attempts = 3
        for attempt in range(1, max_attempts + 1):
            try:
                t_dma0 = time.perf_counter()
                ctx["out_dma"].recvchannel.transfer(out_buf)
                ctx["ip"].write(0x00, 0x01)
                ctx["rx_dma"].sendchannel.transfer(in_sig)
                ctx["prn_dma"].sendchannel.transfer(in_prn)
                t_dma_submit_done = time.perf_counter()

                t_ip_wait0 = time.perf_counter()
                while True:
                    ctrl = int(ctx["ip"].read(0x00))
                    if ((ctrl >> 1) & 1) == 1:
                        break
                    if (time.perf_counter() - t_ip_wait0) > 20.0:
                        raise TimeoutError("Timeout: ap_done non atteint")
                    time.sleep(0.001)
                t_ip_wait1 = time.perf_counter()

                t_dma_wait0 = time.perf_counter()
                ctx["rx_dma"].sendchannel.wait()
                ctx["prn_dma"].sendchannel.wait()
                ctx["out_dma"].recvchannel.wait()
                t_dma_wait1 = time.perf_counter()

                arr = np.asarray(out_buf, dtype=np.float64)
                peak = float(np.max(arr))
                mean_corr = float(np.mean(arr))
                peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
                t_total1 = time.perf_counter()

                dma_submit_s = float(t_dma_submit_done - t_dma0)
                ip_wait_s = float(t_ip_wait1 - t_ip_wait0)
                dma_wait_s = float(t_dma_wait1 - t_dma_wait0)
                dma_total_s = dma_submit_s + dma_wait_s
                kernel_s = ip_wait_s

                return {
                    "method": "fpga",
                    "detected": int(peak_ratio >= ratio_min),
                    "doppler": int(to_s32(ctx["ip"].read(0x10))),
                    "phase": int(ctx["ip"].read(0x20)),
                    "peak": peak,
                    "peak_ratio": peak_ratio,
                    "time_s": kernel_s,
                    "time_kernel_s": kernel_s,
                    "time_prepare_s": float(prep_done - t_total0),
                    "time_dma_submit_s": dma_submit_s,
                    "time_dma_wait_s": dma_wait_s,
                    "time_dma_s": dma_total_s,
                    "time_ip_wait_s": ip_wait_s,
                    "time_end_to_end_s": float(t_total1 - t_total0),
                }
            except Exception as exc:
                last_exc = exc
                print(f"[DMA-RECOVERY] tentative {attempt}/{max_attempts} apres erreur: {exc}")
                reload_needed = attempt >= max_attempts - 1
                _recover_dma_and_ip(ctx, cfg=cfg, reload_overlay=reload_needed)
                if reload_needed:
                    ctx = get_fpga_context(cfg)

        raise RuntimeError(f"Echec FPGA apres recovery DMA: {last_exc}")
    finally:
        in_sig.freebuffer()
        in_prn.freebuffer()
        out_buf.freebuffer()


def run_method(method_name, signal, prn_seq, cfg):
    if method_name == "fpga":
        return fpga_acquisition(signal, prn_seq, cfg)
    raise ValueError(f"Methode inconnue dans ce notebook: {method_name}")


def build_winner_table(df_raw, cfg):
    winner_rows = []
    for (file_name, method_name), group in df_raw.groupby(["file", "method"], dropna=False):
        valid_group = group[group["status"] == "ok"].sort_values("peak_ratio", ascending=False).reset_index(drop=True)
        if valid_group.empty:
            error_row = group.iloc[0]
            winner_rows.append({
                "file": file_name,
                "method": method_name,
                "status": error_row["status"],
                "error_message": error_row.get("error_message", ""),
            })
            continue

        top1 = valid_group.iloc[0]
        doppler_err = int(abs(int(top1["doppler_meas_hz"]) - int(top1["doppler_true_hz"])))
        phase_err = int(phase_error_mod(int(top1["phase_meas_chip"]), int(top1["phase_true_chip"]), int(cfg["nb_phases"])))
        strict_success = int((int(top1["detected"]) == 1) and (int(top1["prn_tested"]) == int(top1["prn_injected"])) and (doppler_err <= int(cfg["doppler_tol_hz"])) and (phase_err <= int(cfg["phase_tol_chip"])))

        winner_rows.append({
            "file": file_name,
            "method": method_name,
            "status": "ok",
            "error_message": "",
            "snr_db": float(top1["snr_db"]),
            "prn_injected": int(top1["prn_injected"]),
            "prn_winner": int(top1["prn_tested"]),
            "doppler_true_hz": int(top1["doppler_true_hz"]),
            "doppler_meas_hz": int(top1["doppler_meas_hz"]),
            "doppler_err_hz": doppler_err,
            "phase_true_chip": int(top1["phase_true_chip"]),
            "phase_meas_chip": int(top1["phase_meas_chip"]),
            "phase_err_chip": phase_err,
            "peak": float(top1["peak"]),
            "peak_ratio": float(top1["peak_ratio"]),
            "time_s": float(top1["time_s"]),
            "strict_success": strict_success,
        })
    return pd.DataFrame(winner_rows)


cfg = get_current_config()
CONFIG = dict(cfg)

signals_dir = Path(cfg["signals_dir"])
prn_dir = Path(cfg["prn_dir"])
signal_files = collect_signal_files(signals_dir, cfg["signal_glob"])
if cfg.get("selected_signal_files"):
    _selected = set(str(x) for x in cfg["selected_signal_files"])
    signal_files = [p for p in signal_files if p.name in _selected]
elif cfg.get("limit_signals"):
    signal_files = signal_files[: int(cfg["limit_signals"])]
if not signal_files:
    raise FileNotFoundError("Aucun signal apres filtrage (verifie le fichier selectionne ou le nombre limite).")

available_prns = get_available_prns(prn_dir)
selected_prns = [int(prn) for prn in cfg["prn_list"] if int(prn) in available_prns] if cfg.get("prn_list") else available_prns

_psm = str(cfg.get("prn_search_mode", "full_grid"))
if _psm == "injected_only":
    _n_prn_per_sig = 1
elif _psm == "injected_plus_decoys":
    _nd = int(cfg.get("pfa_decoy_count", 8))
    _n_prn_per_sig = 1 + min(_nd, max(0, len(selected_prns) - 1))
else:
    _n_prn_per_sig = len(selected_prns)

_n_acq = len(signal_files) * _n_prn_per_sig * len(cfg["run_methods"])
print(f"Acquisitions prevues: {_n_acq}")

export_dir = resolve_export_dir(cfg)
export_dir.mkdir(parents=True, exist_ok=True)
prefix = cfg["export_prefix"]
partial_path = export_dir / f"{prefix}_raw_partial.csv"

new_rows = []
prn_cache = {}
_acq_counter = 0
for index_signal, sig_file in enumerate(signal_files, 1):
    meta = parse_meta_from_header(sig_file)
    prn_injected = int(meta["prn"])
    doppler_true_hz = int(meta["doppler"])
    phase_true_chip = int(meta["ca_shift"] % int(cfg["nb_phases"]))
    snr_db = float(meta["snr"])

    signal = load_signal_pm1(sig_file)
    n_samples = int(cfg["n_samples"])
    signal = np.pad(signal, (0, max(0, n_samples - signal.size)), mode="edge")[:n_samples].astype(np.complex64)

    _others = [int(p) for p in selected_prns if int(p) != int(prn_injected)]
    if _psm == "injected_only" or not _others:
        prns_this_signal = [prn_injected]
    elif _psm == "injected_plus_decoys":
        _k = min(int(cfg.get("pfa_decoy_count", 8)), len(_others))
        _rng = np.random.default_rng(int(cfg.get("pfa_decoy_seed", 42)) + index_signal)
        _pick = _rng.choice(np.array(_others, dtype=np.int64), size=_k, replace=False)
        prns_this_signal = [prn_injected] + sorted(int(x) for x in _pick.tolist())
    else:
        prns_this_signal = selected_prns

    for prn_tested in prns_this_signal:
        prn_global_t0 = time.perf_counter()
        prn_acq_count = 0
        if prn_tested not in prn_cache:
            prn_cache[prn_tested] = load_prn_sequence(prn_dir, prn_tested, n_samples)
        prn_seq = prn_cache[prn_tested]

        try:
            t_e2e0 = time.perf_counter()
            result = run_method("fpga", signal, prn_seq, cfg)
            t_e2e1 = time.perf_counter()
            end_to_end_s = float(t_e2e1 - t_e2e0)
            _acq_counter += 1
            prn_acq_count += 1
            print(
                f"[{_acq_counter}/{_n_acq}] fpga PRN={prn_tested} SNR={snr_db:g} "
                f"DONE global={end_to_end_s:.3f}s"
            )
            new_rows.append({
                "file": sig_file.name,
                "method": "fpga",
                "status": "ok",
                "error_message": "",
                "snr_db": snr_db,
                "prn_injected": prn_injected,
                "prn_tested": int(prn_tested),
                "doppler_true_hz": doppler_true_hz,
                "phase_true_chip": phase_true_chip,
                "detected": int(result["detected"]),
                "doppler_meas_hz": int(result["doppler"]),
                "phase_meas_chip": int(result["phase"]),
                "peak": float(result["peak"]),
                "peak_ratio": float(result["peak_ratio"]),
                "time_s": float(result["time_s"]),
                "time_kernel_s": float(result.get("time_kernel_s", result["time_s"])),
                "time_prepare_s": float(result.get("time_prepare_s", np.nan)),
                "time_dma_submit_s": float(result.get("time_dma_submit_s", np.nan)),
                "time_dma_wait_s": float(result.get("time_dma_wait_s", np.nan)),
                "time_dma_s": float(result.get("time_dma_s", np.nan)),
                "time_ip_wait_s": float(result.get("time_ip_wait_s", np.nan)),
                "time_end_to_end_s": end_to_end_s,
            })
        except Exception as exc:
            _acq_counter += 1
            prn_acq_count += 1
            print(f"[{_acq_counter}/{_n_acq}] fpga PRN={prn_tested} SNR={snr_db:g} ERROR: {exc}")
            new_rows.append({
                "file": sig_file.name,
                "method": "fpga",
                "status": "error",
                "error_message": str(exc),
                "snr_db": snr_db,
                "prn_injected": prn_injected,
                "prn_tested": int(prn_tested),
                "doppler_true_hz": doppler_true_hz,
                "phase_true_chip": phase_true_chip,
                "detected": 0,
                "doppler_meas_hz": np.nan,
                "phase_meas_chip": np.nan,
                "peak": np.nan,
                "peak_ratio": np.nan,
                "time_s": np.nan,
                "time_kernel_s": np.nan,
                "time_prepare_s": np.nan,
                "time_dma_submit_s": np.nan,
                "time_dma_wait_s": np.nan,
                "time_dma_s": np.nan,
                "time_ip_wait_s": np.nan,
                "time_end_to_end_s": np.nan,
            })

        prn_global_elapsed = time.perf_counter() - prn_global_t0
        print(
            f"[PRN-TOTAL] file={sig_file.name} PRN={prn_tested} acquisitions={prn_acq_count} "
            f"global_elapsed={prn_global_elapsed:.3f}s"
        )

        pd.DataFrame(new_rows).to_csv(partial_path, index=False)

raw_results = pd.DataFrame(new_rows)
winner_results = build_winner_table(raw_results, cfg)

raw_results.to_csv(export_dir / f"{prefix}_raw_results.csv", index=False)
winner_results.to_csv(export_dir / f"{prefix}_winner_results.csv", index=False)

print("Exports:")
print(export_dir / f"{prefix}_raw_results.csv")
print(export_dir / f"{prefix}_winner_results.csv")


Acquisitions prevues: 774
[1/774] fpga PRN=25 SNR=-10 DONE global=6.969s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=9.091s
[2/774] fpga PRN=1 SNR=-10 DONE global=2.046s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=1 acquisitions=1 global_elapsed=4.182s
[3/774] fpga PRN=2 SNR=-10 DONE global=2.045s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=2 acquisitions=1 global_elapsed=4.137s
[4/774] fpga PRN=9 SNR=-10 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=9 acquisitions=1 global_elapsed=3.361s
[5/774] fpga PRN=11 SNR=-10 DONE global=2.063s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n10000Hz_ca-804.csv PRN=11 acquisitions=1 global_elapsed=4.164s
[6/774] fpga PRN=13 SNR=-10 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11M

[47/774] fpga PRN=2 SNR=-10 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=2 acquisitions=1 global_elapsed=2.043s
[48/774] fpga PRN=4 SNR=-10 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=4 acquisitions=1 global_elapsed=2.045s
[49/774] fpga PRN=10 SNR=-10 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=10 acquisitions=1 global_elapsed=4.018s
[50/774] fpga PRN=13 SNR=-10 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=13 acquisitions=1 global_elapsed=2.041s
[51/774] fpga PRN=15 SNR=-10 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-n6250Hz_ca-804.csv PRN=15 acquisitions=1 global_elapsed=2.045s
[52/774] fpga PRN=17 SNR=-10 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_d

[93/774] fpga PRN=10 SNR=-10 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p1250Hz_ca-804.csv PRN=10 acquisitions=1 global_elapsed=2.039s
[94/774] fpga PRN=16 SNR=-10 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p1250Hz_ca-804.csv PRN=16 acquisitions=1 global_elapsed=2.042s
[95/774] fpga PRN=18 SNR=-10 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p1250Hz_ca-804.csv PRN=18 acquisitions=1 global_elapsed=4.057s
[96/774] fpga PRN=19 SNR=-10 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p1250Hz_ca-804.csv PRN=19 acquisitions=1 global_elapsed=2.041s
[97/774] fpga PRN=20 SNR=-10 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p1250Hz_ca-804.csv PRN=20 acquisitions=1 global_elapsed=2.042s
[98/774] fpga PRN=23 SNR=-10 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n

[139/774] fpga PRN=13 SNR=-10 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p7500Hz_ca-804.csv PRN=13 acquisitions=1 global_elapsed=2.042s
[140/774] fpga PRN=15 SNR=-10 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p7500Hz_ca-804.csv PRN=15 acquisitions=1 global_elapsed=2.045s
[141/774] fpga PRN=20 SNR=-10 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p7500Hz_ca-804.csv PRN=20 acquisitions=1 global_elapsed=2.043s
[142/774] fpga PRN=24 SNR=-10 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p7500Hz_ca-804.csv PRN=24 acquisitions=1 global_elapsed=2.043s
[143/774] fpga PRN=28 SNR=-10 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_doppler-p7500Hz_ca-804.csv PRN=28 acquisitions=1 global_elapsed=2.044s
[144/774] fpga PRN=30 SNR=-10 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz

[185/774] fpga PRN=16 SNR=-20 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n3750Hz_ca-804.csv PRN=16 acquisitions=1 global_elapsed=2.044s
[186/774] fpga PRN=18 SNR=-20 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n3750Hz_ca-804.csv PRN=18 acquisitions=1 global_elapsed=2.041s
[187/774] fpga PRN=24 SNR=-20 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n3750Hz_ca-804.csv PRN=24 acquisitions=1 global_elapsed=2.041s
[188/774] fpga PRN=28 SNR=-20 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n3750Hz_ca-804.csv PRN=28 acquisitions=1 global_elapsed=2.043s
[189/774] fpga PRN=29 SNR=-20 DONE global=2.044s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n3750Hz_ca-804.csv PRN=29 acquisitions=1 global_elapsed=2.047s
[190/774] fpga PRN=25 SNR=-20 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz

...

[557/774] fpga PRN=19 SNR=10 DONE global=2.062s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p10000Hz_ca-804.csv PRN=19 acquisitions=1 global_elapsed=2.063s
[558/774] fpga PRN=21 SNR=10 DONE global=2.050s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p10000Hz_ca-804.csv PRN=21 acquisitions=1 global_elapsed=2.053s
[559/774] fpga PRN=25 SNR=10 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=2.041s
[560/774] fpga PRN=3 SNR=10 DONE global=2.061s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=3 acquisitions=1 global_elapsed=2.063s
[561/774] fpga PRN=7 SNR=10 DONE global=2.054s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p1250Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=2.056s
[562/774] fpga PRN=16 SNR=10 DONE global=2.054s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10

[603/774] fpga PRN=26 SNR=10 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p6250Hz_ca-804.csv PRN=26 acquisitions=1 global_elapsed=2.046s
[604/774] fpga PRN=25 SNR=10 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=2.043s
[605/774] fpga PRN=7 SNR=10 DONE global=2.047s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=2.048s
[606/774] fpga PRN=14 SNR=10 DONE global=2.049s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=14 acquisitions=1 global_elapsed=2.051s
[607/774] fpga PRN=15 SNR=10 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10_doppler-p7500Hz_ca-804.csv PRN=15 acquisitions=1 global_elapsed=2.042s
[608/774] fpga PRN=16 SNR=10 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p10

[649/774] fpga PRN=25 SNR=20 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=25 acquisitions=1 global_elapsed=2.042s
[650/774] fpga PRN=7 SNR=20 DONE global=2.050s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=2.051s
[651/774] fpga PRN=8 SNR=20 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=8 acquisitions=1 global_elapsed=2.045s
[652/774] fpga PRN=15 SNR=20 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=15 acquisitions=1 global_elapsed=2.042s
[653/774] fpga PRN=16 SNR=20 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-n3750Hz_ca-804.csv PRN=16 acquisitions=1 global_elapsed=2.044s
[654/774] fpga PRN=17 SNR=20 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_d

[695/774] fpga PRN=2 SNR=20 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p0Hz_ca-804.csv PRN=2 acquisitions=1 global_elapsed=2.044s
[696/774] fpga PRN=5 SNR=20 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p0Hz_ca-804.csv PRN=5 acquisitions=1 global_elapsed=2.044s
[697/774] fpga PRN=7 SNR=20 DONE global=2.039s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p0Hz_ca-804.csv PRN=7 acquisitions=1 global_elapsed=2.040s
[698/774] fpga PRN=8 SNR=20 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p0Hz_ca-804.csv PRN=8 acquisitions=1 global_elapsed=2.045s
[699/774] fpga PRN=17 SNR=20 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p0Hz_ca-804.csv PRN=17 acquisitions=1 global_elapsed=2.046s
[700/774] fpga PRN=21 SNR=20 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p0Hz_ca-804.

[742/774] fpga PRN=10 SNR=20 DONE global=2.040s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p5000Hz_ca-804.csv PRN=10 acquisitions=1 global_elapsed=2.042s
[743/774] fpga PRN=11 SNR=20 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p5000Hz_ca-804.csv PRN=11 acquisitions=1 global_elapsed=2.043s
[744/774] fpga PRN=12 SNR=20 DONE global=2.042s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p5000Hz_ca-804.csv PRN=12 acquisitions=1 global_elapsed=2.044s
[745/774] fpga PRN=14 SNR=20 DONE global=2.041s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p5000Hz_ca-804.csv PRN=14 acquisitions=1 global_elapsed=2.043s
[746/774] fpga PRN=22 SNR=20 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p20_doppler-p5000Hz_ca-804.csv PRN=22 acquisitions=1 global_elapsed=2.045s
[747/774] fpga PRN=31 SNR=20 DONE global=2.043s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-p